# External Potential Benchmark: Ellipsoidal Potentials

Wall-clock time and cost relative to falcON self-gravity for vectorized and scalar-loop galpy potential evaluations, as a function of particle count $N$.

Use the checkboxes below to show/hide potential families.

<style>.nbinput { display: none !important; }</style>


In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pathlib

# Load pre-computed benchmark data
data_path = pathlib.Path('../../diagnostics/diagnostics_ellipsoidal_potentials_benchmark_results.npz')
d = np.load(data_path)
N = d['N']
falcon = d['falcon']

GROUPS = {
    'NFW':              [('Spherical NFW (vec)',         'nfw',                       'solid'),
                         ('Triaxial NFW (vec)',           'tnfw_vec',                  'dash'),
                         ('Triaxial NFW (scalar)',        'tnfw_scalar',               'dot')],
    'Hernquist':        [('Spherical Hernquist (vec)',   'hern',                      'solid'),
                         ('Triaxial Hernquist (vec)',     'thern_vec',                 'dash'),
                         ('Triaxial Hernquist (scalar)',  'thern_scalar',              'dot')],
    'Jaffe':            [('Spherical Jaffe (vec)',        'jaffe',                     'solid'),
                         ('Triaxial Jaffe (vec)',         'tjaffe_vec',                'dash'),
                         ('Triaxial Jaffe (scalar)',      'tjaffe_scalar',             'dot')],
    'Power':            [('Power spher. (vec)',           'power_spher',               'solid'),
                         ('Power triaxial (vec)',         'power_triaxial_vec',        'dash'),
                         ('Power triaxial (scalar)',      'power_triaxial_scalar',     'dot')],
    'TwoPower':         [('TwoPower spher. (vec)',        'two_power_spher',           'solid'),
                         ('TwoPower triaxial (vec)',      'two_power_triaxial_vec',    'dash'),
                         ('TwoPower triaxial (scalar)',   'two_power_triaxial_scalar', 'dot')],
    'TriaxialGaussian': [('Triaxial Gaussian (vec)',      'triaxial_gaussian_vec',     'dash'),
                         ('Triaxial Gaussian (scalar)',   'triaxial_gaussian_scalar',  'dot')],
    'PerfectEllipsoid': [('Perfect Ellipsoid (vec)',      'perf_ellip_vec',            'dash'),
                         ('Perfect Ellipsoid (scalar)',   'perf_ellip_scalar',         'dot')],
}

GROUP_COLORS = {
    'NFW':              '#4c9be8',
    'Hernquist':        '#e84c4c',
    'Jaffe':            '#2ecc71',
    'Power':            '#f39c12',
    'TwoPower':         '#9b59b6',
    'TriaxialGaussian': '#1abc9c',
    'PerfectEllipsoid': '#e67e22',
}

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Wall-clock time', 'Cost relative to falcON self-gravity'),
    horizontal_spacing=0.10,
)

trace_categories = []

for col in (1, 2):
    fig.add_trace(go.Scatter(
        x=N, y=falcon if col == 1 else np.ones(len(N)),
        mode='lines+markers',
        line=dict(color='black', width=2.5, dash='solid'),
        marker=dict(symbol='diamond', size=7, color='black'),
        name='falcON self-gravity',
        legendgroup='falcON',
        showlegend=(col == 1),
    ), row=1, col=col)
    trace_categories.append('falcon')

for group_name, lines in GROUPS.items():
    color = GROUP_COLORS[group_name]
    for i, (sub_label, key, dash) in enumerate(lines):
        vals = d[key]
        mask = np.isfinite(vals)
        cat = 'spherical' if dash == 'solid' else ('vectorized' if dash == 'dash' else 'scalar')
        for col in (1, 2):
            y = vals[mask] if col == 1 else (vals / falcon)[mask]
            fig.add_trace(go.Scatter(
                x=N[mask], y=y,
                mode='lines+markers',
                line=dict(color=color, dash=dash, width=1.8),
                marker=dict(size=5, color=color),
                name=sub_label,
                legendgroup=group_name,
                legendgrouptitle_text=group_name if (col == 1 and i == 0) else None,
                showlegend=(col == 1),
            ), row=1, col=col)
            trace_categories.append(cat)

fig.add_hline(y=1.0, line=dict(color='black', dash='dash', width=1), opacity=0.4, row=1, col=2)

n = len(trace_categories)

def vis_for(cats):
    return [True if (trace_categories[i] == 'falcon' or trace_categories[i] in cats) else False
            for i in range(n)]

buttons = [
    dict(label='All',          method='restyle', args=[{'visible': [True] * n}]),
    dict(label='Vectorized',   method='restyle', args=[{'visible': vis_for({'spherical', 'vectorized'})}]),
    dict(label='Unvectorized', method='restyle', args=[{'visible': vis_for({'scalar'})}]),
    dict(label='Spherical',    method='restyle', args=[{'visible': vis_for({'spherical'})}]),
]

fig.update_xaxes(type='log', title_text='Number of particles')
fig.update_yaxes(type='log', title_text='Wall-clock time [s]', row=1, col=1)
fig.update_yaxes(type='log', title_text='Cost relative to falcON', row=1, col=2)

fig.update_layout(
    height=700,
    autosize=True,
    legend=dict(
        orientation='h',
        groupclick='toggleitem',
        tracegroupgap=12,
        font=dict(size=11),
        bgcolor='rgba(255,255,255,0.85)',
        bordercolor='lightgrey',
        borderwidth=1,
        yanchor='top',
        y=-0.18,
        xanchor='center',
        x=0.5,
    ),
    margin=dict(l=60, r=30, t=80, b=60),
    template='simple_white',
    updatemenus=[dict(
        type='buttons',
        direction='right',
        x=0.5, xanchor='center',
        y=1.08, yanchor='top',
        buttons=buttons,
        font=dict(size=12),
        bgcolor='white',
        bordercolor='lightgrey',
        borderwidth=1,
    )],
)

from IPython.display import HTML, display

DIV_ID = "ellipsoidal-benchmark-chart"
plot_html = fig.to_html(full_html=False, include_plotlyjs="cdn", div_id=DIV_ID)
resize_js = """
<script>
(function() {
    var gd = document.getElementById('""" + DIV_ID + """');
    if (!gd) return;
    var timer;
    new ResizeObserver(function() {
        clearTimeout(timer);
        timer = setTimeout(function() { Plotly.Plots.resize(gd); }, 60);
    }).observe(gd.closest('.nboutput') || gd.parentElement || gd);
})();
</script>
"""
display(HTML(plot_html + resize_js))
